# Embeddings NPZ — Inspection & Stats

In [ ]:
import os
import numpy as np
from pathlib import Path

NPZ_PATH = Path("../worldview_unified_final_embeddings.npz")

assert NPZ_PATH.exists(), f"File not found: {NPZ_PATH.resolve()}"
archive = np.load(NPZ_PATH, allow_pickle=False)

ids        = archive["ids"]           # shape (N,)
embeddings = archive["embeddings"]    # shape (N, D)

print(f"File:              {NPZ_PATH.resolve()}")
print(f"File size on disk: {NPZ_PATH.stat().st_size / 1_048_576:.2f} MB")
print(f"Arrays in archive: {list(archive.files)}")
print()
print(f"ids        shape : {ids.shape}  dtype={ids.dtype}")
print(f"embeddings shape : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"  → {embeddings.shape[0]} layers,  {embeddings.shape[1]}-dim vectors")

## Vector statistics

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)

print("Per-vector L2 norms (should be ~1.0 after normalisation in server.py):")
print(f"  min   {norms.min():.6f}")
print(f"  max   {norms.max():.6f}")
print(f"  mean  {norms.mean():.6f}")
print(f"  std   {norms.std():.6f}")
print()

# Value distribution across the full matrix
print("Embedding value distribution (all floats across all rows):")
flat = embeddings.flatten()
print(f"  min   {flat.min():.6f}")
print(f"  max   {flat.max():.6f}")
print(f"  mean  {flat.mean():.6f}")
print(f"  std   {flat.std():.6f}")
print()

# Memory footprint
bytes_in_memory = embeddings.nbytes
print(f"Memory footprint:  {bytes_in_memory / 1_048_576:.2f} MB  (float32 in-RAM)")

## Sample layer IDs

In [ ]:
print("First 10 layer IDs:")
for i, lid in enumerate(ids[:10]):
    print(f"  [{i:4d}]  {lid}")

print()
print("Last 5 layer IDs:")
for i, lid in enumerate(ids[-5:], start=len(ids) - 5):
    print(f"  [{i:4d}]  {lid}")

## Pairwise cosine similarity — sanity check

In [ ]:
# Normalise rows so dot product = cosine similarity
norms_col = np.linalg.norm(embeddings, axis=1, keepdims=True)
normed = embeddings / np.where(norms_col == 0, 1, norms_col)

# Compute pairwise similarities for a small sample (first 200 layers)
sample = normed[:200]
sim_matrix = sample @ sample.T                        # (200, 200)
np.fill_diagonal(sim_matrix, np.nan)                  # exclude self-similarity

print("Pairwise cosine similarity (sample of 200 layers, self excluded):")
print(f"  min   {np.nanmin(sim_matrix):.4f}")
print(f"  max   {np.nanmax(sim_matrix):.4f}  ← most similar pair")
print(f"  mean  {np.nanmean(sim_matrix):.4f}")
print()

# Find the most similar pair
i, j = np.unravel_index(np.nanargmax(sim_matrix), sim_matrix.shape)
print(f"Most similar pair  (sim={sim_matrix[i, j]:.4f}):")
print(f"  [{i}] {ids[i]}")
print(f"  [{j}] {ids[j]}")